In [2]:
pip install settrade-v2

Note: you may need to restart the kernel to use updated packages.


In [4]:
import psycopg2
import pandas as pd
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
from settrade_v2 import Investor
from datetime import datetime

# ==========================================
# 1. ตั้งค่าการเชื่อมต่อ (Configuration)
# ==========================================
DB_HOST = "127.0.0.1"  # หรือ "db" ถ้ารันใน Docker Network เดียวกัน
DB_USER = "postgres"
DB_PASS = "Shifa.326459"
TARGET_DB = "adv_daily"     # ชื่อ Database ใหม่
TARGET_SYMBOL = "ADVANC"

# API Credentials (Sandbox)
APP_ID = "dJNlENF9YYl035kb"
APP_SECRET = "JtBs79Vk4CmwWGVw21aRLeTo2rN/OT7JTcOH8VV+D48="
BROKER_ID = "SANDBOX"
APP_CODE = "SANDBOX"

# ==========================================
# 2. ฟังก์ชันสร้าง Database ใหม่
# ==========================================
def init_database():
    try:
        # เชื่อมต่อ DB กลางเพื่อสร้าง DB ใหม่
        conn = psycopg2.connect(host=DB_HOST, dbname="postgres", user=DB_USER, password=DB_PASS)
        conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
        cur = conn.cursor()
        
        # ลบของเก่าและสร้างใหม่
        cur.execute(f"DROP DATABASE IF EXISTS {TARGET_DB}")
        cur.execute(f"CREATE DATABASE {TARGET_DB}")
        print(f"✅ [SYSTEM] Database '{TARGET_DB}' created successfully.")
        conn.close()
    except Exception as e:
        print(f"❌ [ERROR] Create DB failed: {e}")

# ==========================================
# 3. ฟังก์ชันหลัก (Main Process)
# ==========================================
def main():
    # 3.1 เชื่อมต่อ Database ที่เพิ่งสร้าง
    try:
        conn = psycopg2.connect(host=DB_HOST, dbname=TARGET_DB, user=DB_USER, password=DB_PASS)
        conn.autocommit = True
        cur = conn.cursor()
    except Exception as e:
        print(f"❌ Connection Failed: {e}")
        return

    # 3.2 สร้างตาราง (Schema Design)
    # ตาราง 1: เก็บราคาย้อนหลัง 1000 วัน (OHLCV)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS price_history (
            trade_date DATE NOT NULL,
            symbol VARCHAR(10),
            open NUMERIC,
            high NUMERIC,
            low NUMERIC,
            close NUMERIC,
            volume BIGINT,
            PRIMARY KEY (trade_date, symbol)
        );
    """)

    # ตาราง 2: เก็บข้อมูลเชิงลึก (Deep Data) - เก็บ Log รายวัน
    cur.execute("""
        CREATE TABLE IF NOT EXISTS fundamental_log (
            log_date DATE DEFAULT CURRENT_DATE,
            symbol VARCHAR(10),
            pe NUMERIC,
            pbv NUMERIC,
            eps NUMERIC,
            dividend_yield NUMERIC,
            market_cap NUMERIC,
            market_status VARCHAR(20),
            PRIMARY KEY (log_date, symbol)
        );
    """)
    print("✅ [SCHEMA] Tables 'price_history' and 'fundamental_log' are ready.")

    # 3.3 เชื่อมต่อ API
    try:
        investor = Investor(app_id=APP_ID, app_secret=APP_SECRET, broker_id=BROKER_ID, app_code=APP_CODE, is_auto_queue=False)
        market = investor.MarketData()
        print("✅ [API] Connected to Settrade.")
    except Exception as e:
        print(f"❌ [API Error] {e}")
        return

    # ---------------------------------------------------------
    # PART A: ดึงราคาย้อนหลัง 1,000 วัน (Historical OHLCV)
    # ---------------------------------------------------------
    print(f"\n⏳ [PART A] Fetching 1000-day Price History...")
    try:
        # ใช้ limit=1000 ตามที่ต้องการ
        history = market.get_candlestick(symbol=TARGET_SYMBOL, interval="1d", limit=1000, normalized=True)
        total_records = len(history['time'])
        
        for i in range(total_records):
            ts = int(history['time'][i])
            date_str = datetime.fromtimestamp(ts).strftime('%Y-%m-%d')
            
            sql = """
                INSERT INTO price_history (trade_date, symbol, open, high, low, close, volume)
                VALUES (%s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (trade_date, symbol) DO NOTHING;
            """
            cur.execute(sql, (
                date_str, TARGET_SYMBOL,
                history['open'][i], history['high'][i], history['low'][i], 
                history['close'][i], history['volume'][i]
            ))
        print(f"   ---> ✅ Saved {total_records} days of price history.")
        
    except Exception as e:
        print(f"   ---> ❌ Error Part A: {e}")

    # ---------------------------------------------------------
    # PART B: ดึงข้อมูลเชิงลึก ณ ปัจจุบัน (Deep Data Snapshot)
    # ---------------------------------------------------------
    print(f"\n⏳ [PART B] Fetching Current Deep Data (PE, PBV, EPS)...")
    try:
        # ใช้ get_quote_symbol เพื่อดึงข้อมูลลึก
        quote = market.get_quote_symbol(TARGET_SYMBOL)
        
        # ดึงค่า (ใช้ .get เพื่อกัน Error กรณีค่าว่าง)
        pe = quote.get('pe', 0)
        pbv = quote.get('pbv', 0)
        eps = quote.get('eps', 0)
        div_yield = quote.get('percentYield', 0)
        mkt_cap = quote.get('marketCap', 0) # Sandbox อาจไม่มีค่านี้
        status = quote.get('marketStatus', 'Unknown')

        # บันทึกลงตาราง fundamental_log
        sql_fund = """
            INSERT INTO fundamental_log (log_date, symbol, pe, pbv, eps, dividend_yield, market_cap, market_status)
            VALUES (CURRENT_DATE, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (log_date, symbol) DO UPDATE 
            SET pe=EXCLUDED.pe, pbv=EXCLUDED.pbv, eps=EXCLUDED.eps;
        """
        cur.execute(sql_fund, (TARGET_SYMBOL, pe, pbv, eps, div_yield, mkt_cap, status))
        
        print(f"   ---> ✅ Logged Today's Data:")
        print(f"        PE: {pe} | PBV: {pbv} | EPS: {eps} | Yield: {div_yield}%")

    except Exception as e:
        print(f"   ---> ❌ Error Part B: {e}")

    print("\n🎉 [FINISHED] Process Completed Successfully.")
    conn.close()

if __name__ == "__main__":
    init_database() # สร้าง DB ใหม่ (บรรทัดนี้รันครั้งแรกครั้งเดียว ถ้าจะรันรายวันให้ Comment บรรทัดนี้ออก)
    main()

✅ [SYSTEM] Database 'adv_daily' created successfully.
✅ [SCHEMA] Tables 'price_history' and 'fundamental_log' are ready.
✅ [API] Connected to Settrade.

⏳ [PART A] Fetching 1000-day Price History...
   ---> ✅ Saved 729 days of price history.

⏳ [PART B] Fetching Current Deep Data (PE, PBV, EPS)...
   ---> ✅ Logged Today's Data:
        PE: 24.42 | PBV: 11.23 | EPS: 11.3 | Yield: 3.01%

🎉 [FINISHED] Process Completed Successfully.


In [5]:
import pandas as pd
import psycopg2

# 1. ตั้งค่าการเชื่อมต่อ (Database Configuration)
DB_HOST = "127.0.0.1"
DB_NAME = "adv_daily"     # ชื่อ DB ที่เราเพิ่งสร้าง
DB_USER = "postgres"
DB_PASS = "Shifa.326459"

try:
    # เชื่อมต่อ Database
    conn = psycopg2.connect(
        host=DB_HOST,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASS
    )
    print(f"✅ เชื่อมต่อกับฐานข้อมูล '{DB_NAME}' สำเร็จ!\n")
    print("-" * 50)

    # -------------------------------------------------------
    # เช็กตารางที่ 1: ราคาย้อนหลัง (Price History)
    # -------------------------------------------------------
    print("📊 1. ตรวจสอบตาราง 'price_history' (ราคาย้อนหลัง)")
    
    # ดูจำนวนแถวทั้งหมด (ควรมีประมาณ 1000 แถว)
    count_df = pd.read_sql("SELECT COUNT(*) FROM price_history;", conn)
    total_rows = count_df.iloc[0, 0]
    print(f"👉 จำนวนข้อมูลทั้งหมด: {total_rows} วัน")

    # ดึงข้อมูล 5 วันล่าสุดมาดู
    df_price = pd.read_sql("SELECT * FROM price_history ORDER BY trade_date DESC LIMIT 5;", conn)
    print("👉 ตัวอย่างข้อมูล 5 วันล่าสุด:")
    print(df_price)
    print("-" * 50)

    # -------------------------------------------------------
    # เช็กตารางที่ 2: ข้อมูลเชิงลึก (Fundamental Log)
    # -------------------------------------------------------
    print("📈 2. ตรวจสอบตาราง 'fundamental_log' (PE, PBV, EPS)")
    
    # ดึงข้อมูลทั้งหมดมาดู (ถ้าเพิ่งรันครั้งแรก จะมีแค่ 1 บรรทัดคือของวันนี้)
    df_fund = pd.read_sql("SELECT * FROM fundamental_log ORDER BY log_date DESC;", conn)
    print("👉 ข้อมูล Fundamental ที่บันทึกไว้:")
    print(df_fund)

except Exception as e:
    print(f"❌ เกิดข้อผิดพลาด: {e}")


✅ เชื่อมต่อกับฐานข้อมูล 'adv_daily' สำเร็จ!

--------------------------------------------------
📊 1. ตรวจสอบตาราง 'price_history' (ราคาย้อนหลัง)
👉 จำนวนข้อมูลทั้งหมด: 729 วัน
👉 ตัวอย่างข้อมูล 5 วันล่าสุด:
   trade_date  symbol   open   high    low  close   volume
0  2026-03-31  ADVANC  249.0  249.0  230.0  230.0   130100
1  2026-01-28  ADVANC  350.0  353.0  348.0  352.0  3593192
2  2026-01-27  ADVANC  349.0  354.0  347.0  351.0  7852667
3  2026-01-26  ADVANC  347.0  349.0  344.0  347.0  3931416
4  2026-01-23  ADVANC  343.0  352.0  343.0  351.0  8560177
--------------------------------------------------
📈 2. ตรวจสอบตาราง 'fundamental_log' (PE, PBV, EPS)
👉 ข้อมูล Fundamental ที่บันทึกไว้:
     log_date  symbol     pe    pbv   eps  dividend_yield  market_cap  \
0  2026-01-29  ADVANC  24.42  11.23  11.3            3.01         0.0   

  market_status  
0       OffHour  


C:\Users\Zbook Firefly 14 G8\AppData\Local\Temp\ipykernel_26052\3058731510.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  count_df = pd.read_sql("SELECT COUNT(*) FROM price_history;", conn)
C:\Users\Zbook Firefly 14 G8\AppData\Local\Temp\ipykernel_26052\3058731510.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_price = pd.read_sql("SELECT * FROM price_history ORDER BY trade_date DESC LIMIT 5;", conn)
C:\Users\Zbook Firefly 14 G8\AppData\Local\Temp\ipykernel_26052\3058731510.py:43: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider usi

In [6]:
cur.close()
print("[INFO] Cursor closed.")
conn.close()
print("[INFO] Database connection closed.")

[INFO] Cursor closed.
[INFO] Database connection closed.
